In [0]:
# Objective

# The purpose of this notebook is to validate the quality of the Gold dataset before it is used by:

# Machine Learning
# Power BI
# Business users

# If any validation fails, the issue should be logged for investigation.

In [0]:
# Notebook 10 – Data Quality Framework

#Objective

# Validate the Gold Retention Features dataset before Feature Engineering.

# Validation Checks

# • Row Count Validation
# • Null Value Validation
# • Duplicate Validation
# • Schema Validation
# • Data Type Validation
# • Invalid Date Validation
# • Generate Data Quality Report

# Output

# data_quality_log

In [0]:
# Import Libraries 
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

In [0]:
# Load Gold Dataset 
gold = spark.table("gold_retention_features")

display(gold.limit(5))

CustomerID,JoinDate,Segment,Region,Age,SubscriptionType,TenureDays,AgeGroup,BillingMonth,MonthlyRevenue,TransactionCount,TotalLogins,LifetimeMinutes,AvgSession,LastLoginDate,FirstLoginDate,ActiveDays,LoginFrequency,UsageLevel,SessionCategory,RecentlyActive,TotalTickets,LastTicketDate,OpenTickets,ClosedTickets,ComplaintScore,SupportLevel,OpenTicketRatio,SupportRisk,Churn,ChurnStatus,RetentionFlag,RevenuePerLogin,TicketsPerLogin,EngagementScore,ComplaintRatio,RevenueCategory,LifetimeSegment,LoginCategory,UsageCategory,RiskScore,HighRiskCustomer,CustomerHealth,HealthCategory,LoyaltyScore
100030,2022-06-19,PREMIUM,INDIA,48,MONTHLY,1486,45-59,2024-06,2835.29,1,6,447,74.5,2025-07-16,2024-04-20,452,0.01,LOW,LONG,0,3,2024-07-22,0,3,6,LOW,0.0,LOW,0,ACTIVE,1,472.55,0.5,987.09,2.0,SILVER,LOYAL,LOW,LIGHT,33.36,0,981.09,EXCELLENT,1150.79
100036,2024-09-12,BASIC,USA,43,MONTHLY,670,35-44,2024-01,2912.39,1,7,682,97.42857142857143,2025-05-05,2024-03-16,415,0.02,LOW,LONG,0,3,2025-12-03,1,2,11,LOW,0.33,LOW,0,ACTIVE,1,416.06,0.43,1081.12,3.67,SILVER,ESTABLISHED,LOW,LIGHT,35.49,0,1070.12,EXCELLENT,1011.22
100042,2023-11-02,BASIC,UK,55,MONTHLY,985,45-59,2024-11,3319.43,1,8,865,108.125,2025-09-08,2024-02-16,570,0.01,LOW,LONG,0,3,2024-05-21,0,3,6,LOW,0.0,LOW,0,ACTIVE,1,414.93,0.38,1258.53,2.0,SILVER,LOYAL,LOW,LIGHT,32.28,0,1252.53,EXCELLENT,1196.83
100043,2021-12-10,BASIC,USA,35,MONTHLY,1677,35-44,2024-11,2278.93,1,4,290,72.5,2025-04-07,2024-03-26,377,0.01,LOW,LONG,0,2,2025-05-30,0,2,4,LOW,0.0,LOW,0,ACTIVE,1,569.73,0.5,772.28,2.0,SILVER,LOYAL,LOW,LIGHT,33.52,0,768.28,EXCELLENT,1021.08
100060,2025-01-19,BASIC,CANADA,53,MONTHLY,541,45-59,2025-04,5080.76,1,5,652,130.4,2025-12-03,2024-03-16,627,0.01,LOW,LONG,0,1,2025-08-04,0,1,2,LOW,0.0,LOW,0,ACTIVE,1,1016.15,0.2,1721.83,2.0,GOLD,ESTABLISHED,LOW,LIGHT,29.42,0,1719.83,EXCELLENT,1634.93


In [0]:
# Total Records 
row_count = gold.count()

print("Total Rows :", row_count)

Total Rows : 286893


In [0]:
# Row Count Validation 
if row_count > 0:
    row_status = "PASS"
else:
    row_status = "FAIL"

print("Row Count Validation :", row_status)

Row Count Validation : PASS


In [0]:
# Null Validation 
null_summary = gold.select([

    count(when(col(c).isNull(), c)).alias(c)

    for c in gold.columns

])

display(null_summary)

CustomerID,JoinDate,Segment,Region,Age,SubscriptionType,TenureDays,AgeGroup,BillingMonth,MonthlyRevenue,TransactionCount,TotalLogins,LifetimeMinutes,AvgSession,LastLoginDate,FirstLoginDate,ActiveDays,LoginFrequency,UsageLevel,SessionCategory,RecentlyActive,TotalTickets,LastTicketDate,OpenTickets,ClosedTickets,ComplaintScore,SupportLevel,OpenTicketRatio,SupportRisk,Churn,ChurnStatus,RetentionFlag,RevenuePerLogin,TicketsPerLogin,EngagementScore,ComplaintRatio,RevenueCategory,LifetimeSegment,LoginCategory,UsageCategory,RiskScore,HighRiskCustomer,CustomerHealth,HealthCategory,LoyaltyScore
0,0,0,0,0,0,0,0,4968,0,0,0,0,0,1870,1870,0,0,1870,1870,0,0,38165,0,0,0,0,38165,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Duplicate Validation 
duplicate_rows = row_count - gold.dropDuplicates().count()

print("Duplicate Rows :", duplicate_rows)

duplicate_status = "PASS" if duplicate_rows == 0 else "FAIL"

Duplicate Rows : 0


In [0]:
# Schema Validation 
expected_columns = [

"CustomerID",
"Age",
"Gender",
"Region",
"MonthlyRevenue",
"TotalLogins",
"AvgSession",
"TotalTickets",
"ComplaintScore",
"RiskScore",
"LoyaltyScore",
"Churn"

]

actual_columns = gold.columns

missing_columns = list(set(expected_columns) - set(actual_columns))

extra_columns = list(set(actual_columns) - set(expected_columns))

print("Missing Columns :", missing_columns)

print("Extra Columns :", extra_columns)

schema_status = "PASS" if len(missing_columns)==0 else "FAIL"

Missing Columns : ['Gender']
Extra Columns : ['Segment', 'LifetimeMinutes', 'LoginCategory', 'HealthCategory', 'HighRiskCustomer', 'SubscriptionType', 'LoginFrequency', 'SupportRisk', 'RecentlyActive', 'TransactionCount', 'SupportLevel', 'FirstLoginDate', 'OpenTicketRatio', 'UsageCategory', 'CustomerHealth', 'JoinDate', 'OpenTickets', 'TenureDays', 'AgeGroup', 'ActiveDays', 'BillingMonth', 'ComplaintRatio', 'LastTicketDate', 'ChurnStatus', 'RetentionFlag', 'LifetimeSegment', 'RevenuePerLogin', 'UsageLevel', 'RevenueCategory', 'LastLoginDate', 'EngagementScore', 'TicketsPerLogin', 'ClosedTickets', 'SessionCategory']


In [0]:
# Data Type Validation 
gold.printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- JoinDate: date (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- SubscriptionType: string (nullable = true)
 |-- TenureDays: integer (nullable = true)
 |-- AgeGroup: string (nullable = true)
 |-- BillingMonth: string (nullable = true)
 |-- MonthlyRevenue: double (nullable = true)
 |-- TransactionCount: long (nullable = true)
 |-- TotalLogins: long (nullable = true)
 |-- LifetimeMinutes: long (nullable = true)
 |-- AvgSession: double (nullable = true)
 |-- LastLoginDate: date (nullable = true)
 |-- FirstLoginDate: date (nullable = true)
 |-- ActiveDays: integer (nullable = true)
 |-- LoginFrequency: double (nullable = true)
 |-- UsageLevel: string (nullable = true)
 |-- SessionCategory: string (nullable = true)
 |-- RecentlyActive: integer (nullable = true)
 |-- TotalTickets: long (nullable = true)
 |-- LastTicketDate: date (nullable = true)
 |-- OpenTicke

In [0]:
# Invalid Date Validation 
if "JoinDate" in gold.columns:

    invalid_dates = gold.filter(
        col("JoinDate") > current_date()
    ).count()

else:

    invalid_dates = 0

print("Invalid Dates :", invalid_dates)

Invalid Dates : 0


In [0]:
# Null Count 
total_nulls = 0

for c in gold.columns:

    total_nulls += gold.filter(col(c).isNull()).count()

print("Total Null Values :", total_nulls)

Total Null Values : 88778


In [0]:
# Save Data Quality Log

quality_df.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("data_quality_log")

In [0]:
# Data Quality Summary 
quality_summary = [

(

datetime.now(),

row_count,

duplicate_rows,

total_nulls,

len(missing_columns),

invalid_dates,

row_status,

duplicate_status,

schema_status

)

]

quality_df = spark.createDataFrame(

quality_summary,

[

"RunTime",

"RowCount",

"DuplicateRows",

"NullValues",

"MissingColumns",

"InvalidDates",

"RowValidation",

"DuplicateValidation",

"SchemaValidation"

]

)

display(quality_df)

RunTime,RowCount,DuplicateRows,NullValues,MissingColumns,InvalidDates,RowValidation,DuplicateValidation,SchemaValidation
2026-07-14T20:46:27.034Z,286893,0,88778,1,0,PASS,PASS,FAIL


In [0]:
display(
    spark.table("data_quality_log")
)

RunTime,RowCount,DuplicateRows,NullValues,MissingColumns,InvalidDates,RowValidation,DuplicateValidation,SchemaValidation
2026-07-14T20:38:40.915Z,286893,0,88778,1,0,PASS,PASS,FAIL


In [0]:

# Save report

(
    quality_df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("data_quality_log")
)

In [0]:
# Verify

display(

spark.table("data_quality_log")

)

RunTime,RowCount,DuplicateRows,NullValues,MissingColumns,InvalidDates,RowValidation,DuplicateValidation,SchemaValidation
2026-07-14T20:46:27.034Z,286893,0,88778,1,0,PASS,PASS,FAIL


In [0]:
# Notebook Summary

# Completed

# • Loaded Gold Retention Features
# • Row Count Validation
# • Null Validation
# • Duplicate Validation
# • Schema Validation
# • Data Type Validation
# • Invalid Date Validation
# • Generated Data Quality Report

# Output

# data_quality_log